### SectionE– Gold model, billing and KPIs

In [0]:
dbutils.widgets.text("catalog", "vp_ce_2026_07_1382", "Catalog Name")
catalog = dbutils.widgets.get("catalog")

In [0]:
dbutils.widgets.text("checkpoint_name", "ADHOC", "checkpoint Name")
checkpoint_name = dbutils.widgets.get("checkpoint_name")

In [0]:
%sql
-- ----------------- dim date ----------------
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || ".gold.dim_date") (
  date_key      INT     COMMENT 'Surrogate key in yyyyMMdd integer format, e.g. 20260401',
  calendar_date DATE    COMMENT 'Calendar date for the day',
  day_name      STRING  COMMENT 'Full name of the day of the week (Monday–Sunday)',
  iso_week      INT     COMMENT 'ISO 8601 week-of-year number (1–53)',
  is_weekend    BOOLEAN COMMENT 'TRUE when the day is Saturday or Sunday, FALSE otherwise'
)
USING DELTA;

INSERT INTO IDENTIFIER(:catalog || ".gold.dim_date")
SELECT
  INT(date_format(d, 'yyyyMMdd')) AS date_key,
  d                              AS calendar_date,
  date_format(d, 'EEEE')         AS day_name,
  weekofyear(d)                  AS iso_week,
  dayofweek(d) IN (1, 7)          AS is_weekend
FROM (
  SELECT explode(
    sequence(to_date('2026-04-01'), to_date('2026-04-30'), interval 1 day)
  ) AS d
)
WHERE NOT EXISTS (
  SELECT 1 FROM IDENTIFIER(:catalog || ".gold.dim_date")
);

In [0]:
# -------------------- dim stations --------------------
stations_df = spark.read.table(f"{catalog}.silver.stations")
stations_df.write.mode("append").saveAsTable(f"{catalog}.gold.dim_station")

In [0]:
# -------------------- dim charger --------------------
from pyspark.sql.functions import col

chargers_df = spark.read.table(f"{catalog}.silver.chargers")
stations_df = spark.read.table(f"{catalog}.silver.stations")

dim_charger_df = (
    chargers_df
    .drop("last_changed_snapshot_date")
    .join(
        stations_df.select("station_id", col("city").alias("station_city")),
        on="station_id",
        how="left",
    )
)

dim_charger_df.write.mode("append").saveAsTable(f"{catalog}.gold.dim_charger")

In [0]:
# -------------------- dim member --------------------
from pyspark.sql.functions import row_number, col
from pyspark.sql.window import Window

members_df = spark.read.table(f"{catalog}.silver.members")

dim_member_df = (
    members_df
    .withColumn("member_sk", row_number().over(Window.orderBy(col("member_id"))).cast("long"))
)

dim_member_df.write.mode("append").saveAsTable(f"{catalog}.gold.dim_member")

In [0]:
# ------------------ fact sessions ------------------
from pyspark.sql.functions import (
    col, when, greatest, lit, count, round as _round,
    abs as _abs, date_format, coalesce,
)

# Read silver fact grain + gold dimensions for FK resolution
sessions_df = spark.read.table(f"{catalog}.silver.sessions")
dim_charger  = spark.read.table(f"{catalog}.gold.dim_charger")
dim_member   = spark.read.table(f"{catalog}.gold.dim_member")
tariffs_df   = spark.read.table(f"{catalog}.silver.tariffs")
faults_df    = spark.read.table(f"{catalog}.silver.session_faults")

# --- Resolve member surrogate key from dim_member (gold) ---
member_resolved = (
    sessions_df.select("session_id", "member_id", "session_date")
    .join(dim_member, on="member_id", how="left")
    .filter(
        col("member_id").isNull()
        | (
            (col("session_date") >= col("effective_from"))
            & ((col("session_date") < col("effective_to")) | col("effective_to").isNull())
        )
    )
    .select("session_id", "member_sk", col("plan").alias("plan_at_session"))
)

# --- Fault count per session ---
fault_counts = faults_df.groupBy("session_id").agg(count(lit(1)).alias("fault_count"))

# --- Build fact_sessions ---
fact_df = (
    sessions_df
    # date_key from session_date (yyyyMMdd int) — FK to dim_date, no join needed here
    .withColumn("date_key", date_format(col("session_date"), "yyyyMMdd").cast("int"))
    # join dim_charger (gold) for connector_type → needed to resolve tariff rate
    .join(
        dim_charger.select("charger_id", "connector_type", "station_id"),
        on="charger_id",
        how="left",
    )
    # join tariffs by connector_type and time_band
    .join(
        tariffs_df.select(
            "connector_type", "time_band",
            "rate_per_kwh", "free_idle_minutes", "idle_fee_per_min",
        ),
        on=["connector_type", "time_band"],
        how="left",
    )
    # join resolved member version
    .join(member_resolved, on="session_id", how="left")
    # join fault counts
    .join(fault_counts, on="session_id", how="left")
    # plan_at_session: GUEST for guests (member_id IS NULL)
    .withColumn(
        "plan_at_session",
        when(col("member_id").isNull(), lit("GUEST")).otherwise(col("plan_at_session")),
    )
    # energy_charge = ROUND(energy_kwh * rate_per_kwh)  -- COMPLETED only
    .withColumn(
        "energy_charge",
        when(col("status") == "COMPLETED",
             _round(col("energy_kwh") * col("rate_per_kwh")))
        .otherwise(lit(0.00)),
    )
    # plan_discount = ROUND(energy_charge * rate): BASIC 0%, PRO 10%, FLEET 15%, GUEST 0%
    .withColumn(
        "plan_discount",
        when(col("status") == "COMPLETED",
             _round(
                 col("energy_charge")
                 * when(col("plan_at_session") == "BASIC",  lit(0.0))
                   .when(col("plan_at_session") == "PRO",   lit(0.10))
                   .when(col("plan_at_session") == "FLEET", lit(0.15))
                   .otherwise(lit(0.0))
             ))
        .otherwise(lit(0.00)),
    )
    # idle_fee = max(idle_minutes - free_idle_minutes, 0) * idle_fee_per_min
    .withColumn(
        "idle_fee",
        when(col("status") == "COMPLETED",
             greatest(col("idle_minutes") - col("free_idle_minutes"), lit(0))
             * col("idle_fee_per_min"))
        .otherwise(lit(0.00)),
    )
    # billed_amount = energy_charge - plan_discount + idle_fee
    .withColumn(
        "billed_amount",
        when(col("status") == "COMPLETED",
             col("energy_charge") - col("plan_discount") + col("idle_fee"))
        .otherwise(lit(0.00)),
    )
    # billing_status: OK if |amount_paid - billed_amount| <= 0.50, else MISMATCH; NULL for non-COMPLETED
    .withColumn(
        "billing_status",
        when(col("status") == "COMPLETED",
             when(_abs(col("amount_paid") - col("billed_amount")) <= lit(0.50), lit("OK"))
             .otherwise(lit("MISMATCH")))
        .otherwise(lit(None).cast("string")),
    )
    # fault_count: 0 when no faults recorded
    .withColumn("fault_count", coalesce(col("fault_count"), lit(0)))
    # Final column order — FKs (date_key→dim_date, charger_id→dim_charger,
    # station_id→dim_station, member_sk→dim_member) + measures
    .select(
        "session_id", "date_key", "charger_id", "station_id", "member_id",
        "status", "time_band", "energy_kwh", "duration_minutes", "idle_minutes",
        "amount_paid", "member_sk", "plan_at_session", "rate_per_kwh",
        "energy_charge", "plan_discount", "idle_fee", "billed_amount",
        "billing_status", "fault_count",
    )
)

fact_df.write.mode("append").saveAsTable(f"{catalog}.gold.fact_sessions")


In [0]:
# ------------------ gold.agg_daily_station ------------------
# One row per station per date (first → last session_date, zero-filled gaps).
from pyspark.sql.functions import (
    col, sum as _sum, count as _count, when, lit, coalesce,
    min as _min, max as _max, broadcast, round as _round,
)

fact_df     = spark.read.table(f"{catalog}.gold.fact_sessions")
dim_date    = spark.read.table(f"{catalog}.gold.dim_date")
dim_station = spark.read.table(f"{catalog}.gold.dim_station")
dim_charger = spark.read.table(f"{catalog}.gold.dim_charger")

# ---- date spine: every calendar date from first to last session_date ----
bounds  = fact_df.agg(_min("date_key"), _max("date_key")).collect()[0]
min_key, max_key = bounds[0], bounds[1]

date_spine = (
    dim_date
    .filter((col("date_key") >= lit(min_key)) & (col("date_key") <= lit(max_key)))
    .select("date_key", col("calendar_date").alias("session_date"))
)

# ---- station base: city + charger count (for utilisation denominator) ----
station_base = (
    dim_station.select("station_id", "city")
    .join(
        dim_charger.groupBy("station_id")
                   .agg(_count(lit(1)).alias("charger_count")),
        on="station_id",
        how="left",
    )
)

# ---- full grid: every station × every date (ensures zero-filled days) ----
grid = date_spine.crossJoin(broadcast(station_base))

# ---- daily per-station aggregates from fact_sessions ----
daily = (
    fact_df.groupBy("date_key", "station_id")
    .agg(
        # session counts by status
        _count(when(col("status") == "COMPLETED", lit(1))).alias("completed_sessions"),
        _count(when(col("status") != "COMPLETED",  lit(1))).alias("failed_sessions"),
        # COMPLETED-only measures
        _sum(when(col("status") == "COMPLETED", col("energy_kwh"))
             .otherwise(lit(0.0))).alias("energy_kwh"),
        _sum(when(col("status") == "COMPLETED", col("billed_amount"))
             .otherwise(lit(0.0))).alias("billed_amount"),
        _sum(when(col("status") == "COMPLETED", col("amount_paid"))
             .otherwise(lit(0.0))).alias("amount_paid"),
        # billing + faults
        _count(when(col("billing_status") == "MISMATCH", lit(1))).alias("mismatch_sessions"),
        _sum(coalesce(col("fault_count"), lit(0))).alias("fault_count"),
        # total COMPLETED duration (minutes) for utilisation
        _sum(when(col("status") == "COMPLETED", coalesce(col("duration_minutes"), lit(0)))
             .otherwise(lit(0))).alias("completed_duration"),
    )
)

# ---- left-join grid ← daily, fill zeros, compute utilisation_pct ----
agg_df = (
    grid.join(daily, on=["date_key", "station_id"], how="left")
    .select(
        col("session_date"),
        col("station_id"),
        col("city"),
        coalesce(col("completed_sessions"), lit(0)).alias("completed_sessions"),
        coalesce(col("failed_sessions"),    lit(0)).alias("failed_sessions"),
        _round(coalesce(col("energy_kwh"),    lit(0.0)), 3).alias("energy_kwh"),
        _round(coalesce(col("billed_amount"), lit(0.0)), 2).alias("billed_amount"),
        _round(coalesce(col("amount_paid"),   lit(0.0)), 2).alias("amount_paid"),
        coalesce(col("mismatch_sessions"),   lit(0)).alias("mismatch_sessions"),
        coalesce(col("fault_count"),          lit(0)).alias("fault_count"),
        _round(
            when(coalesce(col("charger_count"), lit(0)) > 0,
                 100.0 * coalesce(col("completed_duration"), lit(0.0))
                 / (coalesce(col("charger_count"), lit(0)) * 1440))
            .otherwise(lit(0.0)), 2,
        ).cast("decimal(5,2)").alias("utilisation_pct"),
    )
)

agg_df.write.mode("append").saveAsTable(f"{catalog}.gold.agg_daily_station")


In [0]:
%sql
CREATE OR REPLACE VIEW IDENTIFIER(:catalog || ".gold.v_kpi_summary") AS
SELECT
  COUNT(CASE WHEN status = 'COMPLETED' THEN 1 END)                         AS completed_sessions,
  COUNT(CASE WHEN status != 'COMPLETED' THEN 1 END)                        AS failed_sessions,
  ROUND(
    COUNT(CASE WHEN status = 'COMPLETED' THEN 1 END) * 100.0
    / NULLIF(
      COUNT(CASE WHEN status = 'COMPLETED' THEN 1 END)
      + COUNT(CASE WHEN status != 'COMPLETED' THEN 1 END), 0
    ), 2
  )                                                                        AS success_rate_pct,
  ROUND(SUM(CASE WHEN status = 'COMPLETED' THEN energy_kwh   ELSE 0 END), 3) AS total_energy_kwh,
  ROUND(SUM(CASE WHEN status = 'COMPLETED' THEN billed_amount ELSE 0 END), 2) AS total_billed,
  ROUND(SUM(CASE WHEN status = 'COMPLETED' THEN amount_paid   ELSE 0 END), 2) AS total_paid,
  ROUND(
    SUM(CASE WHEN status = 'COMPLETED' THEN billed_amount ELSE 0 END)
    - SUM(CASE WHEN status = 'COMPLETED' THEN amount_paid   ELSE 0 END), 2
  )                                                                        AS revenue_leakage,
  COUNT(CASE WHEN billing_status = 'MISMATCH' THEN 1 END)                  AS mismatch_sessions,
  COUNT(DISTINCT CASE WHEN status = 'COMPLETED' THEN member_id END)         AS active_members
FROM IDENTIFIER(:catalog || ".gold.fact_sessions");

In [0]:
# ------------------ ops.reconciliation_results ------------------
from pyspark.sql.functions import col, countDistinct, sum as _sum
from pyspark.sql.types import StructType, StructField, TimestampType, StringType, DecimalType
from decimal import Decimal

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.reconciliation_results (
  run_ts        TIMESTAMP     COMMENT 'Timestamp of the reconciliation run',
  check_name    STRING        COMMENT 'Name of the reconciliation check',
  source_value  DECIMAL(18,3) COMMENT 'Value measured at the source',
  target_value  DECIMAL(18,3) COMMENT 'Value measured at the target',
  status        STRING        COMMENT 'PASS or FAIL'
)
USING DELTA
""")

run_ts = spark.sql("SELECT current_timestamp() AS ts").collect()[0][0]

# ---- Check 1: SESSION_COUNT ----
# Source: distinct standardised session_id in Bronze
# Target: silver.sessions + silver.quarantine_sessions row count
src_sessions = (
    spark.read.table(f"{catalog}.bronze.sessions")
    .select(countDistinct("session_id"))
    .collect()[0][0]
)
tgt_sessions = (
    spark.read.table(f"{catalog}.silver.sessions").count()
    + spark.read.table(f"{catalog}.silver.quarantine_sessions").count()
)
status_1 = "PASS" if src_sessions == tgt_sessions else "FAIL"

# ---- Check 2: ENERGY_KWH ----
# Source: COMPLETED energy_kwh in silver.sessions
# Target: energy_kwh in gold.agg_daily_station
src_energy = (
    spark.read.table(f"{catalog}.silver.sessions")
    .filter(col("status") == "COMPLETED")
    .agg(_sum("energy_kwh"))
    .collect()[0][0]
) or 0.0

tgt_energy = (
    spark.read.table(f"{catalog}.gold.agg_daily_station")
    .agg(_sum("energy_kwh"))
    .collect()[0][0]
) or 0.0

status_2 = "PASS" if abs(float(src_energy) - float(tgt_energy)) < 0.01 else "FAIL"

# ---- Check 3: CURRENT_MEMBERS ----
# Source: distinct member_id in silver.members
# Target: current rows in gold.dim_member
src_members = (
    spark.read.table(f"{catalog}.silver.members")
    .select(countDistinct("member_id"))
    .collect()[0][0]
)
tgt_members = spark.read.table(f"{catalog}.gold.dim_member").count()
status_3 = "PASS" if src_members == tgt_members else "FAIL"

# ---- Append three rows ----
schema = StructType([
    StructField("run_ts",        TimestampType(),    True),
    StructField("check_name",    StringType(),       True),
    StructField("source_value",  DecimalType(18, 3), True),
    StructField("target_value",  DecimalType(18, 3), True),
    StructField("status",        StringType(),       True),
])

results = spark.createDataFrame([
    (run_ts, "SESSION_COUNT",   Decimal(str(src_sessions)), Decimal(str(tgt_sessions)), status_1),
    (run_ts, "ENERGY_KWH",      Decimal(str(src_energy)),   Decimal(str(tgt_energy)),   status_2),
    (run_ts, "CURRENT_MEMBERS", Decimal(str(src_members)),  Decimal(str(tgt_members)),  status_3),
], schema)

results.write.mode("append").saveAsTable(f"{catalog}.ops.reconciliation_results")

In [0]:
# ------------------ ops.checkpoint_counts ------------------
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType,
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.checkpoint_counts (
  checkpoint_name STRING     COMMENT 'Name identifying this checkpoint run',
  table_name      STRING     COMMENT 'Fully-qualified table name (schema.table)',
  row_count       BIGINT     COMMENT 'Number of rows in the table at capture time',
  captured_at     TIMESTAMP  COMMENT 'Timestamp when the count was captured'
)
USING DELTA
""")

tables_to_count = [
    "bronze.sessions",
    "bronze.members",
    "bronze.chargers",
    "silver.sessions",
    "silver.quarantine_sessions",
    "silver.members",
    "silver.chargers",
    "gold.fact_sessions",
    "gold.agg_daily_station",
]

schema = StructType([
    StructField("checkpoint_name", StringType(),    True),
    StructField("table_name",      StringType(),    True),
    StructField("row_count",        LongType(),      True),
    StructField("captured_at",      TimestampType(), True),
])

captured_at = spark.sql("SELECT current_timestamp() AS ts").collect()[0][0]

rows = []
for tbl in tables_to_count:
    cnt = spark.read.table(f"{catalog}.{tbl}").count()
    rows.append((checkpoint_name, f"{catalog}.{tbl}", cnt, captured_at))

checkpoint_df = spark.createDataFrame(rows, schema)
checkpoint_df.write.mode("append").saveAsTable(f"{catalog}.ops.checkpoint_counts")

print(f"{checkpoint_name}: captured {len(rows)} table counts at {captured_at}")


In [0]:
%sql
select * from IDENTIFIER(:catalog || ".ops.checkpoint_counts")